
# B3c — Chronos on Downsampled Weather (Chronos-side notebook)

**Purpose.** Evaluate `amazon/chronos-t5-small` on the exact same
context/target windows as `b3c_panda_downsampled_weather.ipynb` — same
timestamps, same three conditions. Chronos forecasts each channel
**independently** (it has no cross-channel mechanism), so the per-
channel loop below is the correct way to use it, matching every other
Chronos evaluation in this project.

**Environment:** needs a newer `transformers` than Panda's pinned
4.40.2, per the project's established two-environment isolation
pattern. Run this in a separate environment from the Panda notebook,
then bring both CSVs together in
`b3c_analysis_downsampled_weather.ipynb`.

**Critical: the window-construction cells below must be byte-identical
to the ones in the Panda notebook.** They are copy-pasted verbatim
from the same source for exactly that reason — don't edit one without
mirroring the edit in the other.



# B3c — Downsampled-Weather Model Intervention (shared window construction)

**This cell block must be identical, byte-for-byte, in both the Panda
notebook and the Chronos notebook.** The whole point of the design is
that both models see exactly the same context/target pairs, at exactly
the same real-world timestamps, at each resolution — otherwise the
advantage comparison isn't valid.

**Assumption flagged for verification:** this assumes `./ts_data/weather.csv`
(matching `fixed_experiments.ipynb`'s `DATA_DIR` convention) is the same
Jena/Max-Planck file used throughout this project (10-minute native
resolution, a `Date Time` column, and the same 21 numeric channels as
Experiment 8/30/31). Adjust `DATA_DIR` below if your path differs. If
your file uses a different datetime column name, edit `load_weather()`'s
`dt_col` detection before running.


In [13]:
import pandas as pd
import numpy as np

# ---- design constants (fixed before running, per the B3c design) ----
CONTEXT_LEN = 512            # samples, Panda's fixed native context length
N_WINDOWS = 20
NATIVE_H = 96                 # native (10-min) horizon -> 16h physical span
HOURLY_H_FIXED_SAMPLE = 96     # fixed-sample-horizon convention at hourly res (~4 days physical)
HOURLY_H_FIXED_PHYSICAL = 16   # fixed-physical-horizon convention at hourly res (16h, matches native)
DOWNSAMPLE_FACTOR = 6          # matches Experiment 31; hourly matches ETTh1's native sampling rate
SEED = 0

DATA_DIR = "./ts_data"   # matches fixed_experiments.ipynb's convention -- adjust if needed
WEATHER_PATH = f"{DATA_DIR}/weather.csv"


In [14]:
def load_weather(path=WEATHER_PATH):
    df = pd.read_csv(path)
    dt_col = 'Date Time' if 'Date Time' in df.columns else df.columns[0]
    # Format detection rather than a hardcoded assumption: the raw Jena
    # download uses dd.mm.yyyy (dot-separated, day/month ambiguous), but
    # preprocessed versions of this file are often already ISO yyyy-mm-dd
    # (dash-separated, unambiguous). Forcing dayfirst=True on an
    # already-ISO string breaks it (pandas tries %Y-%d-%m instead of
    # %Y-%m-%d). Check which one this file actually is before parsing.
    sample = str(df[dt_col].dropna().iloc[0])
    if '.' in sample:
        df[dt_col] = pd.to_datetime(df[dt_col], dayfirst=True)   # raw Jena dd.mm.yyyy
    else:
        df[dt_col] = pd.to_datetime(df[dt_col])                  # ISO yyyy-mm-dd, unambiguous
    df = df.set_index(dt_col).sort_index()
    # Duplicate timestamps break get_indexer(method='nearest') in make_window
    # below (it requires a unique index). Real-world Jena-style exports
    # sometimes carry a handful of repeated or DST-adjacent rows; drop
    # duplicates here rather than letting the failure surface three
    # function calls downstream where it's harder to diagnose.
    n_before = len(df)
    df = df[~df.index.duplicated(keep='first')]
    n_dropped = n_before - len(df)
    if n_dropped > 0:
        print(f'load_weather: dropped {n_dropped} duplicate-timestamp rows (kept first occurrence)')
    numeric_df = df.select_dtypes(include=[np.number])
    return numeric_df

def build_hourly(df_native):
    # Simple stride decimation (every DOWNSAMPLE_FACTOR-th native sample),
    # matching Experiment 31's downsampling method exactly -- NOT an hourly
    # average. This preserves point-sample character, consistent with how
    # ETTh1 itself is point-sampled rather than hour-averaged. If Experiment
    # 31's notebook used a different downsampling method (e.g. mean pooling),
    # switch this to match it exactly for comparability with that result.
    return df_native.iloc[::DOWNSAMPLE_FACTOR]

def valid_start_range(df_native, df_hourly):
    # Hourly context (512 hourly steps, ~21 days) spans far longer in real
    # time than native context (512 native steps, ~3.6 days), so hourly's
    # context requirement is the binding constraint on the left margin.
    # The right margin only needs to fit the larger of the three horizons
    # in physical time (hourly_H96_fixedsample = 4 days is the largest).
    min_start_time = df_hourly.index[CONTEXT_LEN]
    max_start_time = df_hourly.index[-1] - pd.Timedelta(hours=HOURLY_H_FIXED_SAMPLE)
    valid_native = df_native.loc[min_start_time:max_start_time]
    return valid_native.index

def get_window_start_timestamps(df_native, df_hourly, n_windows=N_WINDOWS):
    valid_idx = valid_start_range(df_native, df_hourly)
    positions = np.linspace(0, len(valid_idx) - 1, n_windows, dtype=int)
    return valid_idx[positions]

def make_window(df, start_ts, context_len, horizon_len):
    start_idx = df.index.get_indexer([start_ts], method='nearest')[0]
    if start_idx - context_len < 0 or start_idx + horizon_len > len(df):
        return None, None
    context = df.iloc[start_idx - context_len:start_idx]
    target = df.iloc[start_idx:start_idx + horizon_len]
    if len(context) < context_len or len(target) < horizon_len:
        return None, None
    return context.values.astype(np.float32), target.values.astype(np.float32)

def build_all_windows(weather_path=WEATHER_PATH):
    df_native = load_weather(weather_path)
    df_hourly = build_hourly(df_native)
    starts = get_window_start_timestamps(df_native, df_hourly)

    conditions = {
        'native_H96':             (df_native, CONTEXT_LEN, NATIVE_H),
        'hourly_H96_fixedsample': (df_hourly, CONTEXT_LEN, HOURLY_H_FIXED_SAMPLE),
        'hourly_H16_fixedphys':   (df_hourly, CONTEXT_LEN, HOURLY_H_FIXED_PHYSICAL),
    }

    windows = {name: [] for name in conditions}
    dropped = {name: 0 for name in conditions}
    for start_ts in starts:
        for name, (df, ctx_len, hor_len) in conditions.items():
            ctx, tgt = make_window(df, start_ts, ctx_len, hor_len)
            if ctx is None:
                dropped[name] += 1
                windows[name].append(None)
            else:
                windows[name].append((ctx, tgt))

    for name, n_dropped in dropped.items():
        if n_dropped > 0:
            print(f'WARNING: {name} dropped {n_dropped}/{N_WINDOWS} windows '
                  f'(insufficient context/horizon margin at that timestamp)')

    channels = list(df_native.columns)
    return windows, channels

windows, channels = build_all_windows()  # uses WEATHER_PATH = f"{DATA_DIR}/weather.csv" by default
print('Channels:', channels)
print('Conditions:', list(windows.keys()))
for name, wlist in windows.items():
    n_valid = sum(1 for w in wlist if w is not None)
    print(f'  {name}: {n_valid}/{N_WINDOWS} valid windows')


load_weather: dropped 1 duplicate-timestamp rows (kept first occurrence)
Channels: ['p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)', 'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)', 'rain (mm)', 'raining (s)', 'SWDR (W/m�)', 'PAR (�mol/m�/s)', 'max. PAR (�mol/m�/s)', 'Tlog (degC)', 'OT']
Conditions: ['native_H96', 'hourly_H96_fixedsample', 'hourly_H16_fixedphys']
  native_H96: 20/20 valid windows
  hourly_H96_fixedsample: 20/20 valid windows
  hourly_H16_fixedphys: 20/20 valid windows


## Load Chronos

Matches your `fixed_experiments.ipynb` loading code, with one change:
`torch_dtype` is now device-conditional. `bfloat16` is fine on GPU but
risks two separate CPU failures -- some ops aren't implemented for
bfloat16 on CPU, and even where it works, `.numpy()` on a bfloat16
tensor raises `TypeError: Got unsupported ScalarType BFloat16` (numpy
has no bfloat16 type). Since you're running locally, this defaults to
float32 automatically; it'll still use bfloat16 if you ever run this on
a CUDA machine.

In [15]:
import torch
from chronos import ChronosPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device == 'cuda' else torch.float32
print(f"Device: {device}, dtype: {dtype}")

chronos_pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map=device,
    torch_dtype=dtype,
)
print("Loaded amazon/chronos-t5-small.")


Device: cpu, dtype: torch.float32


C:\Users\user\panda_env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded amazon/chronos-t5-small.


## Forecast helper

Per-channel univariate forecasting. Mirrors `panda_forecast`'s final
form exactly: returns **normalized** predictions (never denormalize
before computing error -- Experiment 8 and every other advantage number
in this project is in normalized units, not raw physical units), and
guards near-zero-variance channels (e.g. `rain (mm)` during a dry
window) the same way, so a single degenerate channel can't blow up its
own error to astronomical values.

In [16]:
def chronos_forecast(context_tc, horizon):
    mean = context_tc.mean(axis=0, keepdims=True)
    std = context_tc.std(axis=0, keepdims=True)

    degenerate = std < 1e-6
    safe_std = np.where(degenerate, 1.0, std)
    context_norm = (context_tc - mean) / safe_std

    T, C = context_norm.shape
    preds = np.zeros((horizon, C), dtype=np.float32)
    for c in range(C):
        series = torch.tensor(context_norm[:, c], dtype=torch.float32)
        forecast = chronos_pipeline.predict(
            inputs=series, prediction_length=horizon, num_samples=1,
        )
        samples = forecast[0].to(torch.float32).numpy()
        preds[:, c] = np.median(samples, axis=0)

    return preds, mean, safe_std, degenerate

## Run Chronos on all three conditions, save raw predictions + per-window MAE

In [17]:
import traceback

context, target = windows['native_H96'][0]
mean = context.mean(axis=0, keepdims=True)
std = context.std(axis=0, keepdims=True)
degenerate = std < 1e-6
safe_std = np.where(degenerate, 1.0, std)
context_norm = (context - mean) / safe_std

series = torch.tensor(context_norm[:, 0], dtype=torch.float32)
try:
    forecast = chronos_pipeline.predict(context=series, prediction_length=NATIVE_H, num_samples=1)
    print('OK, forecast type:', type(forecast))
    print('OK, forecast shape:', forecast.shape)
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_5188\142170083.py", line 12, in <module>
    forecast = chronos_pipeline.predict(context=series, prediction_length=NATIVE_H, num_samples=1)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: ChronosPipeline.predict() got an unexpected keyword argument 'context'


In [18]:
import inspect
print(inspect.signature(chronos_pipeline.predict))

(inputs: Union[torch.Tensor, List[torch.Tensor]], prediction_length: Optional[int] = None, num_samples: Optional[int] = None, temperature: Optional[float] = None, top_k: Optional[int] = None, top_p: Optional[float] = None, limit_prediction_length: bool = False) -> torch.Tensor


In [9]:
context, target = windows['native_H96'][0]
pred_norm, mean, safe_std, degenerate = chronos_forecast(context, NATIVE_H)
print('pred_norm shape:', pred_norm.shape, '-- should be (96, 21)')

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

pred_norm shape: (96, 21) -- should be (96, 21)


In [19]:
import os

os.makedirs('b3c_raw_predictions', exist_ok=True)
results = []
n_degenerate = 0

for condition, wlist in windows.items():
    horizon = {'native_H96': NATIVE_H,
               'hourly_H96_fixedsample': HOURLY_H_FIXED_SAMPLE,
               'hourly_H16_fixedphys': HOURLY_H_FIXED_PHYSICAL}[condition]
    for window_idx, w in enumerate(wlist):
        if w is None:
            continue
        context, target = w
        try:
            pred_norm, mean, safe_std, degenerate = chronos_forecast(context, horizon)
        except Exception as e:
            print(f'FAILED: {condition} window {window_idx}: {type(e).__name__}: {e}')
            continue

        n_degenerate += degenerate.sum()
        target_norm = (target - mean) / safe_std

        np.savez(
            f'b3c_raw_predictions/chronos_{condition}_w{window_idx:02d}.npz',
            context=context, target=target,
            forecast_norm=pred_norm, mean=mean, std=safe_std, degenerate=degenerate,
        )

        abs_err = np.abs(pred_norm - target_norm)
        for c_idx, channel_name in enumerate(channels):
            results.append({
                'model': 'chronos', 'condition': condition, 'window_idx': window_idx,
                'channel': channel_name, 'mae': float(abs_err[:, c_idx].mean()),
                'degenerate': bool(degenerate[0, c_idx]),
            })

chronos_df = pd.DataFrame(results)
chronos_df.to_csv('b3c_chronos_predictions.csv', index=False)
print(f'Degenerate (near-zero-std) channel-windows guarded: {n_degenerate}')
print(chronos_df.groupby('condition')['mae'].agg(['mean', 'median', 'count']))


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

Degenerate (near-zero-std) channel-windows guarded: 8
                            mean    median  count
condition                                        
hourly_H16_fixedphys    0.465664  0.329043    420
hourly_H96_fixedsample  0.761951  0.610773    420
native_H96              0.840979  0.666094    420


## Sanity check against Experiment 8

`native_H96` here should land close to Experiment 8's Weather H=96
Chronos MAE (0.8115 at n=20).

In [20]:
native_mae = chronos_df[chronos_df.condition == 'native_H96']['mae'].mean()
print(f'This notebook, native_H96, mean Chronos MAE across channels: {native_mae:.4f}')
print('Experiment 8 reference (Weather H=96, n=20): 0.8115')
print('If these differ substantially, check num_samples/aggregation choices before trusting B3c.')


This notebook, native_H96, mean Chronos MAE across channels: 0.8410
Experiment 8 reference (Weather H=96, n=20): 0.8115
If these differ substantially, check num_samples/aggregation choices before trusting B3c.


In [12]:
test_results = []
for window_idx in range(5):  # 5 windows, not all 20 -- enough to see a real signal, ~12x cheaper than the full run
    context, target = windows['native_H96'][window_idx]
    mean = context.mean(axis=0, keepdims=True)
    std = context.std(axis=0, keepdims=True)
    safe_std = np.where(std < 1e-6, 1.0, std)
    context_norm = (context - mean) / safe_std
    target_norm = (target - mean) / safe_std

    for n_samp in [1, 20]:
        preds = np.zeros((NATIVE_H, context.shape[1]), dtype=np.float32)
        for c in range(context.shape[1]):
            series = torch.tensor(context_norm[:, c], dtype=torch.float32)
            forecast = chronos_pipeline.predict(inputs=series, prediction_length=NATIVE_H, num_samples=n_samp)
            samples = forecast[0].to(torch.float32).numpy()
            preds[:, c] = np.median(samples, axis=0) if n_samp > 1 else samples[0]
        window_mae = np.abs(preds - target_norm).mean()
        test_results.append({'window_idx': window_idx, 'num_samples': n_samp, 'mae': window_mae})

test_df = pd.DataFrame(test_results)
print(test_df.pivot(index='window_idx', columns='num_samples', values='mae'))
print()
print('num_samples=1 median MAE:', test_df[test_df.num_samples==1]['mae'].median())
print('num_samples=20 median MAE:', test_df[test_df.num_samples==20]['mae'].median())

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 
We recommend keeping prediction length <= 64. The quality of l

num_samples        1         20
window_idx                     
0            0.801636  0.857412
1            2.466619  2.409292
2            0.739779  0.624640
3            1.324204  1.203797
4            0.769797  0.670532

num_samples=1 median MAE: 0.8016355
num_samples=20 median MAE: 0.857412
